## This is the code to train the model and acquire influence for Number of Features Experiment

**Default Code**:   
The current default code is a runnable sample. It runs on the synthetic dataset generated with sklearn's make_classification function. The default version code provides the synthetic dataset with 16500 samples and 10 features in total. The separation is set to 1 to make sure the dataset is distinguishable by the model. All features are set to be informative to ensure they are of equal importance. The dataset has only two labels, so it is a binary classification problem. The default setting will then generate the training set and test set from the pool. The default training sample size is 5000, and the test size is 500. The number of features is set to 10 plus the 42 noise features. The model in default will be a Simple FeedForward Neural Network constructed by TensorFlow. The Influence Estimation methods we provide by default are the Influence Function and TracIn. If you simply press 'play', the default code will generate ranked influence lists for both Influence Function and TracIn with respect to the above mentioned setting in the root directory. The result lists could then be fed into other analyses.

**By default, the only thing changing here is about modifying the noise features added. This will be explained in this code. For the other common information, Please refer to the base code for more detailed explanation.**

**Guideline**:  
Read in / Construct Datasets -> **Choose the Noise Feature Size** -> Pre-processing -> Model Training -> Influence Estimation -> Store the Ranked Influence lists -> **Change the Noise Feature Size and Repeat all the process** -> ... -> **After all the training and estimation, feed the results into the analysis code**  (Remember to change the file name in the last block to save lists in different settings.)

# Import Area

Here is the area to place all the import codes. You don't need to change here unless you want to customise in later sections.

In [285]:
import tensorflow as tf
import tensorflow_datasets as tfds
import keras
from keras.utils import to_categorical
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [286]:
from keras import Sequential
from keras.layers import Dense, BatchNormalization, Dropout
from keras.losses import CategoricalCrossentropy
from keras.optimizers import Adam

In [287]:
from deel.influenciae.common import InfluenceModel, ExactIHVP
from deel.influenciae.influence import FirstOrderInfluenceCalculator
from deel.influenciae.utils import ORDER
from deel.influenciae.trac_in import TracIn

In [288]:
import random
from keras.optimizers import SGD

In [289]:
from sklearn.metrics import pairwise_distances
from sklearn.manifold import MDS
import seaborn as sns
import matplotlib.pyplot as plt

In [290]:
from sklearn.datasets import make_classification

# Dataset Construction Area

**You can use any dataset you wish here, either regression or classification. But in default, since we are using influenciae's IF and TC method, make sure they are split into train and test sets, and then stored as tensorflow dataset format. If you only want to change the dataset, you can only change the code in the first two blocks to read in/ generate your own dataset. But remember to have features X and target y before going to the third block. Also, if you wish to everything on your own, remember to add id inside the dataset.** Since our default code is for classification, the regression might need a lot of changes in all the following sections.


**Input**: Dataset chosen(Usually in Features X and Target y format)  
**Output**: Tensorflow format Train and Test Set  
**Guideline**: Input -> Turn into Dataframe and add ID -> Pre-Processing -> Change the format to Tensorflow -> Output

The default code now produces a synthetic dataset with 16500 pool, 10 features with binary classification problems. The later options will turn that into a 10 features + 42 noise features, 8000 train set and 500 test set sample. Both sets will then be turned into TensorFlow format and will wait for training.

**The most important thing in this code is changing the noise features. In this experiment, all the other things are fixed, but the number of noise features is changing to test on different number of features.**

1. Set your default setting here. train_pool + test_size = Total Dataset Size. train_sizes determines the later subset data. In this experiment, the sample size will always be set to 5000. Sep to make sure the dataset is distinguishable.

In [291]:
train_pool = 16000
test_size = 500
train_sizes=[1000,2000,3000,4000,5000,6000,7000,8000,9000,10000]
n_features=10
seed=42
sep = 1

2. Construct the Synthetic Dataset with Make Classification here. **Could replace this with other datasets with X and y.**

In [292]:
total_samples = train_pool + test_size

X, y = make_classification(n_samples=total_samples,
                           n_features=n_features,
                           n_informative=n_features,
                           n_clusters_per_class = 1,
                           n_redundant=0,
                           n_repeated=0,
                           n_classes=2,
                           class_sep=sep,
                           random_state=seed)

3. **The Feature Noises are added at this step.** n_noise determines the total number of noise features generated. The k from Ks will be the exact noise features added to the dataset during each run. **Change KS[i] to modify the noise features.**

In [293]:
KS = [10,14,18,22,26,30,34,38,42]
n_noise = 90
rng = np.random.RandomState(seed) 
X_noise = rng.normal(loc=0.0, scale=1.0, size=(X.shape[0], n_noise))

k= KS[8]

X = np.hstack([X, X_noise[:, :k]])
# k = 0

4. Turn the X and y into dataframe for easy further processing. Add ID column to easy retrieve samples within IF/TC.

In [294]:
df = pd.DataFrame(X, columns=[f'feature_{i+1}' for i in range(n_features+k)])
df['label'] = y
df['id'] = np.arange(1, len(df) + 1)
print(df)

       feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
0      -2.462785   1.784031   2.750230   1.983940   0.331680   1.174118   
1      -3.425860   3.091771  -0.996399   1.152936   0.487537  -0.794771   
2      -3.666989   1.969536   1.389431   2.247002   2.385491   0.830315   
3      -4.224882  -1.480716   3.421315   1.025147   0.097334   4.042792   
4      -4.244179  -5.092764   3.780899  -3.046639  -3.902753  -3.343005   
...          ...        ...        ...        ...        ...        ...   
16495  -1.738303  -2.368518   0.184498  -2.834851  -1.781499   0.763730   
16496  -2.446645   0.955477   4.436585  -0.664160   0.935989  -4.083520   
16497  -2.289426   0.550712   2.845396   3.849258  -0.076908   2.120920   
16498  -5.918908   2.077796   2.784047   1.993381  -4.229283   4.286199   
16499  -2.527862  -1.756233   3.466682  -5.149739   2.350795   0.277100   

       feature_7  feature_8  feature_9  feature_10  ...  feature_45  \
0      -2.195223  -1.469477 

In [295]:
df_train_pool = df.iloc[:train_pool].reset_index(drop=True)
df_test = df.iloc[train_pool:].reset_index(drop=True)

In [296]:
print(df_train_pool.head())
print(df_test.head())

   feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
0  -2.462785   1.784031   2.750230   1.983940   0.331680   1.174118   
1  -3.425860   3.091771  -0.996399   1.152936   0.487537  -0.794771   
2  -3.666989   1.969536   1.389431   2.247002   2.385491   0.830315   
3  -4.224882  -1.480716   3.421315   1.025147   0.097334   4.042792   
4  -4.244179  -5.092764   3.780899  -3.046639  -3.902753  -3.343005   

   feature_7  feature_8  feature_9  feature_10  ...  feature_45  feature_46  \
0  -2.195223  -1.469477   1.106515    0.137798  ...    0.822545   -1.220844   
1   2.273718   2.553062   1.129106   -5.716311  ...    0.586857    2.190456   
2   1.854461   3.752189   1.141437   -4.276726  ...   -0.315269    0.758969   
3  -2.073007  -2.305776   0.674058   -0.978083  ...   -0.020902    0.117327   
4  -3.954134   1.157540   1.570334   -3.043344  ...    1.179440   -0.469176   

   feature_47  feature_48  feature_49  feature_50  feature_51  feature_52  \
0    0.208864   -1.95

5. Here, we choose the subset of the full dataset. By setting features_to_test, we have the subset feature size. By changing nested_train_dfs, we have different sample sizes. Here the feature size will always be n_features+k since we use all the features plus noise features to form the subset dataset.

In [297]:
features_to_test = n_features+k
selected_features = [f'feature_{i+1}' for i in range(features_to_test)]

nested_train_dfs = [df_train_pool.iloc[:size].reset_index(drop=True) for size in train_sizes]
nested_train_dfs = [df[ selected_features + ['label', 'id'] ].copy()for df in nested_train_dfs]

df_test = df_test[ selected_features + ['label', 'id'] ].copy()

core_1000_ids = nested_train_dfs[0]['id'].tolist()

In [298]:
train_df = nested_train_dfs[4]

In [299]:
X_train = train_df.drop(columns=["label"])
y_train = train_df["label"]
IDs = X_train["id"].values.reshape(-1, 1).astype(np.float32)
IDs = IDs  / 1e10

X_train = X_train.drop(columns=["id"]).values.astype(np.float32)
X_train = np.hstack((X_train, IDs))
y_train = to_categorical(y_train.values,num_classes=2)

print(X_train)

[[-2.4627850e+00  1.7840309e+00  2.7502301e+00 ...  7.3846656e-01
   1.7136829e-01  1.0000000e-10]
 [-3.4258599e+00  3.0917711e+00 -9.9639922e-01 ... -1.5506635e+00
   6.8562977e-02  2.0000000e-10]
 [-3.6669888e+00  1.9695363e+00  1.3894312e+00 ...  2.3146586e+00
  -1.8672652e+00  3.0000000e-10]
 ...
 [ 1.0106713e+00  3.4303145e+00 -2.0520663e+00 ... -5.1954083e-02
  -4.7618221e-02  4.9980002e-07]
 [-1.7018661e+00  1.4068928e+00  2.0173202e+00 ...  1.3813001e+00
   8.0218202e-01  4.9990001e-07]
 [-1.7224492e+00  1.0327566e+00  4.8909622e-01 ... -1.4136842e-01
   1.0220490e+00  5.0000000e-07]]


In [300]:
test_df = df_test
X_test = test_df.drop(columns=["label"])
y_test = test_df["label"]
IDs = X_test["id"].values.reshape(-1, 1).astype(np.float32)
IDs = IDs  / 1e10

X_test = X_test.drop(columns=["id"]).values.astype(np.float32)
X_test = np.hstack((X_test, IDs))
y_test = to_categorical(y_test.values,num_classes=2)

print(X_test)

[[ 8.8689077e-01 -8.0902457e-01  5.2976408e+00 ...  6.0584897e-01
  -1.0062188e+00  1.6000999e-06]
 [ 1.7393481e+00  1.2651124e+00 -1.3206822e+00 ...  1.6550098e-01
   9.2904598e-01  1.6002000e-06]
 [-8.2115895e-01 -2.9575674e-02  1.0195701e+00 ... -9.0736997e-01
   2.9648161e-01  1.6003000e-06]
 ...
 [-2.2894258e+00  5.5071223e-01  2.8453963e+00 ... -8.7264138e-01
   1.2606430e+00  1.6498000e-06]
 [-5.9189086e+00  2.0777962e+00  2.7840469e+00 ...  1.7713413e+00
  -1.2217505e+00  1.6499000e-06]
 [-2.5278618e+00 -1.7562330e+00  3.4666820e+00 ...  1.2274666e+00
   5.9780866e-01  1.6499999e-06]]


6.1 (Optional) The following code below can display the samples distribution. Uncomment them to acquire the distribution plot.

In [301]:
# X_all = np.vstack([X_train, X_test])

# y_train_1d = np.argmax(y_train, axis=1)
# y_test_1d  = np.argmax(y_test, axis=1)
# y_all_1d = np.hstack([y_train_1d, y_test_1d])

# D = pairwise_distances(X_all) 

In [302]:
# X_mds = MDS(n_components=2, dissimilarity='precomputed', random_state=0).fit_transform(D)

In [303]:
# sns.scatterplot(x=X_mds[:,0], y=X_mds[:,1], hue=y_all_1d)
# plt.title("MDS – preserves original distances")

7. Now we have the train_ds and test_ds for training

In [304]:
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))

# Training Area

**Could modify the model as you wish here. Again, in default, influenciae relies on TensorFlow, so use the TensorFlow model if you only want to change the model. Remember: Store the InfluenceModel into the model_list with the loss function. The InfluenceModel will be used to obtain influence later. If you don't change the estimation methods, then the final output at this step shall always be the model_list**

**Input**:Train and Test Set from Data Construction Section   
**Output**: Model List  
**Guideline**: Input -> Define the Model and Hyperparameters -> Train the Model -> Output

Always remember to train the model, get the influence model and store that in model list, unless you wish to change the estimation methods.

The default code now use the train and test set generated from the last section to train the model. The default hyperparameters are: 300 Epochs, Simple FeedForward Neural Network, CategoricalCrossEntropy Loss function, SGD optimizer. Within each epoch, the current model will be turned into an Influence Model and stored inside a model list. After the training, the model list will be passed to next section for influence estimation.

In [305]:
from tensorflow.keras.regularizers import l2

1. **Could modify the model as you wish here as long as it is tensorflow.** Just remember: Store the InfluenceModel into the model_list with the loss function

In [306]:
seed_value = 42
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

model = Sequential([
    Dense(16, activation='relu', input_shape=(X_train.shape[1],)),  
    BatchNormalization(momentum=0.9),
    Dropout(0.0),
    Dense(8, activation='relu'),
    Dense(y_train.shape[1])
])
loss_fn = CategoricalCrossentropy(from_logits=True)
optimizer = SGD(learning_rate=0.001, momentum=0.9)
model.compile(loss=loss_fn, optimizer=optimizer, metrics=['accuracy'])

epochs = 300
unreduced_loss_fn = CategoricalCrossentropy(from_logits=True, reduction=tf.keras.losses.Reduction.NONE)
model_list = []
model_list.append(InfluenceModel(model, start_layer=-1, loss_function=unreduced_loss_fn))
for i in range(epochs):
  model.fit(train_ds.batch(256), epochs=1, validation_data=test_ds.batch(256), verbose=2)
  model_list.append(InfluenceModel(model, start_layer=-1, loss_function=unreduced_loss_fn))
base_loss, acc = model.evaluate(test_ds.batch(32), verbose=2)
print(base_loss)

20/20 - 1s - loss: 0.9182 - accuracy: 0.4832 - val_loss: 0.8008 - val_accuracy: 0.5020 - 637ms/epoch - 32ms/step
20/20 - 0s - loss: 0.7629 - accuracy: 0.5408 - val_loss: 0.6820 - val_accuracy: 0.5740 - 380ms/epoch - 19ms/step
20/20 - 0s - loss: 0.6606 - accuracy: 0.6080 - val_loss: 0.6120 - val_accuracy: 0.6400 - 198ms/epoch - 10ms/step
20/20 - 0s - loss: 0.6023 - accuracy: 0.6618 - val_loss: 0.5685 - val_accuracy: 0.6880 - 259ms/epoch - 13ms/step
20/20 - 0s - loss: 0.5632 - accuracy: 0.6978 - val_loss: 0.5366 - val_accuracy: 0.7240 - 335ms/epoch - 17ms/step
20/20 - 0s - loss: 0.5329 - accuracy: 0.7286 - val_loss: 0.5111 - val_accuracy: 0.7700 - 246ms/epoch - 12ms/step
20/20 - 0s - loss: 0.5073 - accuracy: 0.7528 - val_loss: 0.4896 - val_accuracy: 0.7840 - 189ms/epoch - 9ms/step
20/20 - 0s - loss: 0.4847 - accuracy: 0.7710 - val_loss: 0.4704 - val_accuracy: 0.7940 - 176ms/epoch - 9ms/step
20/20 - 0s - loss: 0.4642 - accuracy: 0.7884 - val_loss: 0.4527 - val_accuracy: 0.8020 - 187ms/epo

# Influence Estimation Area

**Again, you could use other influence analysis methods rather than IF/TC. You can also use any other Influence Function or TracIn implementation. Just Remember: 1. Make sure the package is unform throughout the framework. 2. Generate a Ranked influence list for each Influence Function and TracIn; Only the ranked influence list could be fed into the following analysis code.**

**The default code now use the model list, train set and test set to estimate the influence, and produce a ranked influence list for both IF and TC. The results are then saved in the root directory.**

**Input**:Model list from Training section, Train and Test Set from Data Construction Section   
**Output**: Two ranked Influence Lists for IF and TC.  
**Guideline**: Input -> Influence Estimation Methods -> Influence Matrix -> Output

1. Influence Function: Here we use the influenciae package. The following code will directly generate the influence list. If you wish to have the matrix, just use influence_matrix.

In [307]:
train_ids = []
test_ids = []
train_samples_np = np.array([x.numpy() for x, y in train_ds])
train_ids = [round(sample[-1] * 1e10) for sample in train_samples_np]

In [308]:
num_test_samples = len(test_ds)
num_train_samples = len(train_ids)
test_ids = []

influence_model = model_list[-1]
ihvp_calculator = ExactIHVP(influence_model, train_ds.batch(16))
influence_calculator = FirstOrderInfluenceCalculator(influence_model, train_ds, ihvp_calculator)

influence_matrix = np.zeros((num_test_samples, num_train_samples))

samples_to_explain = test_ds.take(num_test_samples).batch(1)
explanation_ds = influence_calculator.top_k(samples_to_explain, train_ds.batch(16), k=num_train_samples, order=ORDER.DESCENDING)
for test_idx,((sample, label), top_k_values, top_k_samples) in enumerate(explanation_ds.as_numpy_iterator()):
    test_sample_id = round(sample[0][-1] * 1e10)
    test_ids.append(test_sample_id)
    influential_ids = [round(s[-1] * 1e10) for s in top_k_samples[0]] 
    influence_scores = top_k_values[0]
    id_to_index = {train_id: idx for idx, train_id in enumerate(train_ids)}
    for inf_id, score in zip(influential_ids, influence_scores):
        if inf_id in id_to_index:
            influence_matrix[test_idx, id_to_index[inf_id]] = score

flattened_row = np.median(influence_matrix, axis=0).reshape(1, -1)
df = pd.DataFrame({'Train_ID': train_ids, 'Score': flattened_row.flatten()})
print(df)

      Train_ID         Score
0            1  2.357133e-05
1            2  1.674585e-06
2            3  4.272400e-06
3            4  5.903342e-06
4            5  3.480961e-08
...        ...           ...
4995      4996  2.909733e-04
4996      4997  4.479201e-05
4997      4998  1.221973e-03
4998      4999  8.324805e-04
4999      5000  1.114894e-05

[5000 rows x 2 columns]


2. TracIn: Here we use the influenciae package. The following code will directly generate the influence list. If you wish to have the matrix, just use TracIn_matrix.

In [309]:
num_test_samples = len(test_ds)
num_train_samples = len(train_ids)
test_ids = []

TracIn_matrix = np.zeros((num_test_samples, num_train_samples))
influence_calculator = TracIn(
    model_list, 0.001
)
samples_to_explain = test_ds.take(num_test_samples).batch(1)
explanation_ds = influence_calculator.top_k(samples_to_explain, train_ds.batch(16), k=num_train_samples, order=ORDER.DESCENDING)
for test_idx,((sample, label), top_k_values, top_k_samples) in enumerate(explanation_ds.as_numpy_iterator()):
    test_sample_id = round(sample[0][-1] * 1e10)
    test_ids.append(test_sample_id)
    influential_ids = [round(s[-1] * 1e10) for s in top_k_samples[0]] 
    influence_scores = top_k_values[0]
    id_to_index = {train_id: idx for idx, train_id in enumerate(train_ids)}
    for inf_id, score in zip(influential_ids, influence_scores):
        if inf_id in id_to_index:
            TracIn_matrix[test_idx, id_to_index[inf_id]] = score

flattened_row = np.median(TracIn_matrix, axis=0).reshape(1, -1)
TracIn_df = pd.DataFrame({'Train_ID': train_ids, 'Score': flattened_row.flatten()})
print(TracIn_df)

      Train_ID         Score
0            1 -9.573729e-09
1            2  4.012854e-10
2            3  7.274286e-10
3            4 -2.930519e-10
4            5 -4.064052e-13
...        ...           ...
4995      4996  2.755397e-08
4996      4997  1.171494e-08
4997      4998  1.593702e-07
4998      4999  8.155676e-08
4999      5000 -1.408271e-09

[5000 rows x 2 columns]


3. Here we turn both influence lists to the ranked influence lists and then store them for further processing.

In [310]:
df_sorted = df.sort_values(by="Score", ascending=False).reset_index(drop=True)

TracIn_sorted = TracIn_df.sort_values(by="Score", ascending=False).reset_index(drop=True)

In [311]:
TracIn_sorted.to_csv("TC_Feature_Column_10_42.csv",index = False)
df_sorted.to_csv("IF_Feature_Column_10_42.csv",index = False)